# Downlaod data

In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [3]:
repo_fnd = read_repo_data('katjaweb', 'MLOps-Fake-News-Detection')
dtc_llm_zoomcamp = read_repo_data('DataTalksClub', 'llm-zoomcamp')

print(f"FND documents: {len(repo_fnd)}")
print(f"FND documents: {len(dtc_llm_zoomcamp)}")

FND documents: 1
FND documents: 48


In [4]:
import re
text = repo_fnd[0]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [5]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [6]:
fnd_chunks = []

for doc in repo_fnd:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    fnd_chunks.extend(chunks)

In [7]:
fnd_chunks[1]

{'start': 1000,
 'chunk': 'ndividuals to critically assess the credibility of the information they encounter online.\n\nThe fake news detection web-service is built using Flask, a web application framework for Python, as the backend server. The server processes user input, specifically the text of a news article, and passes it through a machine-learning model to determine whether the news is real or fake. To provide transparency, the model also outputs a probability score, ensuring users understand that predictions are not always absolute. The service is deployed using AWS Elastic Beanstalk, which manages the infrastructure, scaling, and deployment of the application. The web service can also be tested using a user interface built with Flask, running on a development server.\n\nTo enhance accuracy, standard natural language processing (NLP) techniques were applied to preprocess the text, including tokenization and stopword removal. Additionally, new numerical features were created and 

In [8]:
dtc_llm_zoomcamp_chunks = []

for doc in dtc_llm_zoomcamp:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    dtc_llm_zoomcamp_chunks.extend(chunks)

In [9]:
dtc_llm_zoomcamp_chunks[0]

{'start': 0,
 'chunk': '# Module 1: Introduction\n \nIn this module, we will learn what LLM and RAG are and\nimplement a simple RAG pipeline to answer questions about \nthe FAQ Documents from our Zoomcamp courses\n\nWhat we will do: \n\n* Index Zoomcamp FAQ documents\n    * DE Zoomcamp: https://docs.google.com/document/d/19bnYs80DwuUimHM65UV3sylsCn2j1vziPOwzBwQrebw/edit\n    * ML Zoomcamp: https://docs.google.com/document/d/1LpPanc33QJJ6BSsyxVg-pWNMplal84TdZtq10naIhD8/edit\n    * MLOps Zoomcamp: https://docs.google.com/document/d/12TlBfhIiKtyBv8RnsoJR6F72bkPDGEvPOItJIxaEzE0/edit\n* Create a Q&A system for answering questions about these documents \n\n## 1.1 Introduction to LLM and RAG\n\n<a href="https://www.youtube.com/watch?v=Q75JgLEXMsM&list=PL3MmuxUbc_hIB4fSqLy_0AfTjVLpgjV3R">\n  <img src="https://markdown-videos-api.jorgenkh.no/youtube/Q75JgLEXMsM">\n</a>\n\n* LLM\n* RAG\n* RAG architecture\n* Course outcome\n\n\n## 1.2 Preparing the Environment\n\n<a href="https://www.youtube.com

# Add search

## lexical search

In [10]:
from minsearch import Index

index_fnd = Index(
    text_fields=["chunk"],
    keyword_fields=[]
)

index_fnd.fit(fnd_chunks)

In [11]:
index_llm_zoomcamp = Index(
    text_fields=["chunk"],
    keyword_fields=[]
)

index_llm_zoomcamp.fit(dtc_llm_zoomcamp_chunks)

## Sematnic search

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
from tqdm.auto import tqdm
import numpy as np

fnd_embeddings = []

for d in tqdm(fnd_chunks):
    text = d['chunk']
    v = embedding_model.encode(text)
    fnd_embeddings.append(v)

fnd_embeddings = np.array(fnd_embeddings)

  0%|          | 0/17 [00:00<?, ?it/s]

In [14]:
from minsearch import VectorSearch

fnd_vindex = VectorSearch()
fnd_vindex.fit(fnd_embeddings, fnd_chunks)

In [15]:
llm_embeddings = []

for d in tqdm(dtc_llm_zoomcamp_chunks):
    text = d['chunk']
    v = embedding_model.encode(text)
    llm_embeddings.append(v)

llm_embeddings = np.array(llm_embeddings)

  0%|          | 0/260 [00:00<?, ?it/s]

In [16]:
llm_vindex = VectorSearch()
llm_vindex.fit(llm_embeddings, dtc_llm_zoomcamp_chunks)

In [17]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = llm_vindex.search(q)

In [18]:
results[0]

{'start': 2000,
 'chunk': '",\n  \'section\': \'General course-related questions\',\n  \'question\': \'Course - Can I still join the course after the start date?\',\n  \'course\': \'data-engineering-zoomcamp\'},\n {\'text\': \'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.\',\n  \'section\': \'General course-related questions\',\n  \'question\': \'Course - Can I follow the course after it finishes?\',\n  \'course\': \'data-engineering-zoomcamp\'},\n {\'text\': "The purpose of this document is to capture frequently asked technical questions\\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours\'\' live.1\\nSubscribe to course public Google Calendar (it works from Desktop only).\\nRe

## Hybrid search

In [19]:
def text_search(query):
    return index_llm_zoomcamp.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return llm_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results

In [20]:
query = 'Can I still enroll ther course?'
hybrid_search(query)

[{'start': 0,
  'chunk': "Question: I just discovered the couse. can i still enrol\n\nContext:\n\nCourse - Can I still join the course after the start date?\nYes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.\n\nEnvironment - Is Python 3.9 still the recommended version to use in 2024?\nYes, for simplicity (of troubleshooting against the recorded videos) and stability. [source]\nBut Python 3.10 and 3.11 should work fine.\n\nHow can we contribute to the course?\nStar the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.\n\nAre we still using the NYC Trip data for January 2021? Or are we using the 2022 data?\nWe will use the same data, as the project will essentially remain the same as last year’s. The data is available here\n\nDocker-Compose - 

# Agents and tools

In [25]:
import openai
from dotenv import load_dotenv
load_dotenv()

openai_client = openai.OpenAI()

In [22]:
# build search function
def text_search(query):
    return index_llm_zoomcamp.search(query, num_results=5)

In [23]:
# describe the search function
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [26]:
# use search function
system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

In [27]:
# look at output
response.output

[ResponseFunctionToolCall(arguments='{"query":"can I join the course now"}', call_id='call_Pi3ojCl0ZJ1BOyJPasGR2fav', name='text_search', type='function_call', id='fc_008061a79acb77d50069d74c95d9688194a4127aef443c9e74', namespace=None, status='completed')]

In [28]:
# invoke function with arguments from output
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}

In [29]:
# make the LLM stateful
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

Yes, you can still join the course after the start date. However, you should note that some deadlines may apply, especially regarding homework and final projects.

Once you register, you will have access to the materials and can progress at your own pace. Here are some steps you should take:

1. **Register for the course** before it starts.
2. **Join the course channels** on Slack and Telegram for announcements.
3. **Explore the course materials** and prepare by reviewing the prerequisites.

If you're ready, feel free to register!


In [30]:
# refine system prompt
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

In [34]:
#write docstring to describe function
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return index_llm_zoomcamp.search(query, num_results=5)

In [35]:
# define an agent with PydanticAI
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)

In [36]:
# run the agent
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

In [37]:
# get detailed information about the agents reasoning and actions. Look at user prompt
result.new_messages()[0]

ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 4, 9, 6, 55, 11, 522216, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 9, 6, 55, 11, 522445, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provide accurate answers.\nIf the search doesn't return relevant results, let the user know and provide general guidance.", run_id='019d7105-fea1-7777-be2f-a8dce7991447')

In [38]:
# Take a look at model's response
result.new_messages()[1]

ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"join course enrollment deadlines"}', tool_call_id='call_BNMALMs0UoFv0EvB7o58e9Ej')], usage=RequestUsage(input_tokens=161, output_tokens=17, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 4, 9, 6, 55, 13, 186160, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://api.openai.com/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 4, 9, 6, 55, 12, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-DSdaCQSPmvqgn73oMKuWOvabS4gsg', finish_reason='tool_call', run_id='019d7105-fea1-7777-be2f-a8dce7991447')

In [39]:
# results returned by the tool
result.new_messages()[2]

ModelRequest(parts=[ToolReturnPart(tool_name='text_search', content=[{'start': 1000, 'chunk': ' embedding model returns are already normalized: their length is 1.0.\n\nYou can check that by using the `norm` function:\n\n```python\nimport numpy as np\nnp.linalg.norm(q)\n```\n\nWhich means that we can simply compute the dot product between\ntwo vectors to learn the cosine similarity between them.\n\nFor example, if you compute the cosine of the query vector with itself, the result will be 1.0:\n\n```python\nq.dot(q)\n```\n\n## Q2. Cosine similarity with another vector\n\nNow let\'s embed this document:\n\n```python\ndoc = \'Can I still join the course after the start date?\'\n```\n\nWhat\'s the cosine similarity between the vector for the query\nand the vector for the document?\n\n* 0.3\n* 0.5\n* 0.7\n* 0.9\n\n## Q3. Ranking by cosine\n\nFor Q3 and Q4 we will use these documents:\n\n```python\ndocuments = [{\'text\': "Yes, even if you don\'t register, you\'re still eligible to submit the

In [40]:
# model response
result.new_messages()[3]

ModelResponse(parts=[TextPart(content="Yes, you can still join the course even if it has already started. However, there will be important deadlines for submitting assignments and projects that you should be aware of, so it's a good idea to not delay your registration.\n\nHere are some key details to consider:\n- You can register before the course officially starts. The course is set to begin on **January 15, 2024**, at **17:00**.\n- Make sure to join the relevant communication channels like Slack and Telegram to stay updated on announcements and course materials.\n\nIf you are interested, act quickly to ensure you meet any additional requirements!")], usage=RequestUsage(input_tokens=2532, output_tokens=124, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 4, 9, 6, 55, 16, 68119, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url=